# 03 · `agent-framework` sin MCP

[Microsoft Agent Framework](https://github.com/microsoft/agent-framework) es una
librería para construir agentes: te da el **loop de function-calling** (preguntar al
LLM, ver si pidió llamar una tool, ejecutarla, devolverle el resultado, repetir hasta
que responda en texto) ya resuelto, sobre cualquier proveedor de chat (OpenAI, Azure,
Ollama, etc.).

Es el framework que usa `jub-agent/agent/jub_agent.py`. Antes de conectarlo a MCP
(notebook 04), vamos a usarlo con una tool Python normal y corriente — para separar
"qué hace agent-framework" de "qué aporta MCP" (son cosas independientes: MCP es *de
dónde* vienen las tools, agent-framework es *quién* decide llamarlas).

Requisitos:
```bash
pip install "httpx<1" "mcp==1.28.1" --pre agent-framework-ollama
ollama pull qwen2.5:1.5b   # o el modelo que prefieras
ollama serve               # en otra terminal, si no corre ya como servicio
```

> Nota: los pines de `httpx` y `mcp` no son capricho — sin ellos, `--pre`
> deja que pip elija versiones incompatibles (ver el README de esta carpeta,
> sección "Cómo correr los notebooks", para el detalle de por qué).


## Las piezas: `ChatClient` y `Agent`

- Un **`ChatClient`** (`OllamaChatClient`, `OpenAIChatClient`, ...) sabe hablar con un
  proveedor de LLM concreto.
- `chat_client.as_agent(name=, instructions=, tools=[...])` construye un **`Agent`**:
  un `ChatClient` + instrucciones de sistema + una lista de tools.
- `Agent` es un *async context manager* (`async with agent: ...`) y expone
  `await agent.run(mensaje)`, que corre el loop completo y devuelve un resultado con
  `.text`.

Una tool para `agent-framework` es, en el caso más simple, **una función Python
normal**: el framework lee su firma (tipos) y su docstring para generar el JSON Schema
que el modelo necesita, igual que hace un servidor MCP con `@mcp.tool()` — es el mismo
mecanismo, solo que aquí la función vive en el proceso del agente en vez de en un
servidor aparte.


In [ ]:
from typing import Annotated

from agent_framework.ollama import OllamaChatClient

OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = "qwen2.5:1.5b"


def calcular_propina(
    cuenta: Annotated[float, "Monto total de la cuenta"],
    porcentaje: Annotated[float, "Porcentaje de propina, ej. 15 para 15%"],
) -> float:
    """Calcula cuánto dejar de propina dado el monto de una cuenta y un porcentaje."""
    return round(cuenta * porcentaje / 100, 2)


agent = OllamaChatClient(host=OLLAMA_URL, model=OLLAMA_MODEL).as_agent(
    name="AgenteDePropinas",
    instructions=(
        "Eres un asistente que ayuda a calcular propinas. Usa siempre la tool "
        "disponible para calcular; no hagas la cuenta a mano ni la inventes."
    ),
    tools=[calcular_propina],
    
)

async with agent:
    resultado = await agent.run(
        "Si mi cuenta fue de 850 pesos, ¿cuánto debería dejar de propina al 12%?"
    )
    print(resultado.text)


Deberías dejar de propina por 102 pesos en tu cuenta de 850 pesos a un 12%.


## Qué pasó por debajo

1. `agent.run(...)` mandó tu mensaje al modelo junto con el JSON Schema de
   `calcular_propina` (generado desde el type hint `Annotated` y el docstring).
2. El modelo respondió pidiendo llamar `calcular_propina(cuenta=850, porcentaje=12)` en
   vez de responder texto directamente.
3. `agent-framework` ejecutó la función Python de verdad, tomó el resultado (`102.0`).
4. Se lo devolvió al modelo como resultado de la tool call.
5. El modelo generó la respuesta final en texto, usando ese número real (no uno
   alucinado).

Este loop — pedir, decidir, ejecutar, responder — es exactamente lo que también hace
cuando la tool viene de un servidor MCP en vez de ser una función local: **la única
diferencia va a ser de dónde sale la tool**, no cómo se usa. Eso es lo que vemos ahora.

**Siguiente:** [`04_conectando_agente_a_mcp.ipynb`](04_conectando_agente_a_mcp.ipynb)
